In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import time
import colorsys
import random

# --- Configuration ---
MAX_POINTS = 150        # Max points to keep in history (higher = smaller triangles, more chaos)
TRAIL_decay = 0.5       # How fast points disappear
OPACITY = 0.6           # Transparency of the fractal layer (0.0 to 1.0)
COLOR_SPEED = 0.05      # How fast colors cycle

# --- MediaPipe Setup ---
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def media_pipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def get_hand_coords(landmarks, image_shape):
    h, w, c = image_shape
    coords = []
    # We only take a subset of landmarks to reduce lag and make the mesh sharper
    # Tips of fingers and palm center
    indices = [0, 4, 8, 12, 16, 20] 
    for i in indices:
        lm = landmarks.landmark[i]
        coords.append((int(lm.x * w), int(lm.y * h)))
    return coords

def get_rainbow_color(value, brightness=1.0):
    # Generates a color tuple based on a float value (0.0 - 1.0)
    r, g, b = colorsys.hsv_to_rgb(value % 1.0, 0.8, brightness)
    return (int(b * 255), int(g * 255), int(r * 255))

def rect_contains(rect, point):
    if point[0] < rect[0]: return False
    if point[1] < rect[1]: return False
    if point[0] > rect[0] + rect[2]: return False
    if point[1] > rect[1] + rect[3]: return False
    return True

def draw_delaunay_chaos(img, points, time_offset):
    """
    Draws the chaotic Delaunay triangulation based on point cloud.
    """
    h, w, _ = img.shape
    rect = (0, 0, w, h)
    
    # Create an instance of Subdiv2D
    subdiv = cv2.Subdiv2D(rect)
    
    # Insert points into subdiv
    for p in points:
        # Ensure points are within frame bounds to prevent errors
        if rect_contains(rect, p):
            subdiv.insert(p)
            
    triangleList = subdiv.getTriangleList()
    
    # Create a separate overlay for transparency
    overlay = img.copy()
    
    for t in triangleList:
        pt1 = (int(t[0]), int(t[1]))
        pt2 = (int(t[2]), int(t[3]))
        pt3 = (int(t[4]), int(t[5]))
        
        # Filter out triangles that are essentially outside the frame 
        # (Subdiv2D creates giant triangles connecting to corners)
        if rect_contains(rect, pt1) and rect_contains(rect, pt2) and rect_contains(rect, pt3):
            
            # --- CHAOS COLORING LOGIC ---
            # Calculate centroid to determine color
            centroid_x = (pt1[0] + pt2[0] + pt3[0]) / 3
            centroid_y = (pt1[1] + pt2[1] + pt3[1]) / 3
            
            # Map position + time to Hue
            hue = (centroid_x / w) + (centroid_y / h) + time_offset
            
            # Map triangle size to brightness (smaller = brighter/sharper)
            area = abs(0.5 * ((pt1[0]*(pt2[1]-pt3[1])) + (pt2[0]*(pt3[1]-pt1[1])) + (pt3[0]*(pt1[1]-pt2[1]))))
            brightness = min(1.0, 5000 / (area + 1)) 
            
            color = get_rainbow_color(hue, brightness=brightness)
            
            # Draw the filled triangle
            pts = np.array([pt1, pt2, pt3], np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.fillPoly(overlay, [pts], color)
            
            # Draw the wireframe (optional, makes it look more technical)
            # cv2.polylines(overlay, [pts], True, (255,255,255), 1, cv2.LINE_AA)

    # Apply transparency
    cv2.addWeighted(overlay, OPACITY, img, 1 - OPACITY, 0, img)

# --- Main Execution ---

point_history = []
cap = cv2.VideoCapture(0)
global_time = 0.0

# Set camera resolution (optional, helps with performance)
cap.set(3, 1280)
cap.set(4, 720)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Flip frame for mirror effect
        frame = cv2.flip(frame, 1)
        
        # Run detection
        image, results = media_pipe_detection(frame, holistic)
        
        h, w, c = image.shape
        new_points = []
        
        # Extract Hand Landmarks
        if results.right_hand_landmarks:
            new_points.extend(get_hand_coords(results.right_hand_landmarks, image.shape))
        if results.left_hand_landmarks:
            new_points.extend(get_hand_coords(results.left_hand_landmarks, image.shape))
            
        # Add new points to history
        # We add some random jitter to new points to increase the "Chaos" factor
        for p in new_points:
            jitter_x = random.randint(-10, 10)
            jitter_y = random.randint(-10, 10)
            point_history.append((p[0] + jitter_x, p[1] + jitter_y))
            
        # Manage History Size (First-In-First-Out)
        if len(point_history) > MAX_POINTS:
            point_history = point_history[-MAX_POINTS:]
            
        # Update global time for color cycling
        global_time += COLOR_SPEED
        
        # --- DRAW THE CHAOS ---
        # Only draw if we have enough points to form a triangle (3 points)
        if len(point_history) >= 3:
            draw_delaunay_chaos(image, point_history, global_time)
            
        # Show the result
        cv2.imshow('Voronoi Chaos Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

2026-02-08 23:04:13.483738: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-08 23:04:13.507089: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-08 23:04:13.507113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-08 23:04:13.507951: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-08 23:04:13.512292: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-08 23:04:13.512912: I tensorflow/core/platform/cpu_feature_guard.cc:1

error: OpenCV(4.11.0) /io/opencv/modules/imgproc/src/subdivision2d.cpp:288: error: (-211:One of the arguments' values is out of range)  in function 'locate'


: 